# 🌽 MayaVoice — Fine-tuning Llama 3.1-8B para Idiomas Mayas
## Sprint 1: MVP con Unsloth + QLoRA en Google Colab

**Objetivo:** Fine-tune Llama 3.1-8B-Instruct para traducción ES↔Maya usando Unsloth con QLoRA de 4-bit.

**Dataset:** 23,738 pares paralelos (12 idiomas mayas, ambas direcciones)

**Issue:** [#6 - Primer fine-tuning con Unsloth en Colab Pro](https://github.com/DanielRegaladoUMiami/mayavoice-llm/issues/6)


In [ ]:
# ============================================================
# 1. INSTALACIÓN (solo ejecutar una vez por sesión de Colab)
# ============================================================
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install sacrebleu


In [ ]:
# ============================================================
# 2. CONFIGURACIÓN
# ============================================================
import json
import random
import torch
from pathlib import Path

# --- Configuración principal ---
CONFIG = {
    "base_model": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "warmup_ratio": 0.1,
    "num_train_epochs": 3,
    "seed": 42,
    "output_dir": "./mayavoice-lora",
    "hub_model_id": "DanielRegaladoUMiami/mayavoice-llama3.1-8b-lora",
}

random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


In [ ]:
# ============================================================
# 3. CARGAR MODELO BASE CON UNSLOTH
# ============================================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["base_model"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

# Agregar LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["seed"],
)

# Contar parámetros entrenables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parámetros entrenables: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


In [ ]:
# ============================================================
# 4. CARGAR DATOS
# ============================================================
# 📌 INSTRUCCIONES: Sube los archivos train.jsonl y val.jsonl a Colab
# Opción 1: Subir manualmente con el botón de archivos
# Opción 2: Desde Google Drive:
#   from google.colab import drive
#   drive.mount('/content/drive')
#   DATA_DIR = "/content/drive/MyDrive/mayavoice-data/"

DATA_DIR = "./"  # Cambiar si los datos están en otro lugar

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

train_data = load_jsonl(f"{DATA_DIR}/train.jsonl")
val_data = load_jsonl(f"{DATA_DIR}/val.jsonl")

print(f"Train: {len(train_data)} ejemplos")
print(f"Val:   {len(val_data)} ejemplos")
print(f"\nEjemplo:")
print(json.dumps(train_data[0], indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 5. FORMATEAR DATOS PARA LLAMA 3.1 INSTRUCT
# ============================================================
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

def format_example(example):
    """Convierte un ejemplo Alpaca al formato de chat de Llama 3.1."""
    messages = [
        {"role": "system", "content": "Eres MayaVoice, un asistente especializado en idiomas mayas de Guatemala. Traduces con precisión y respetas la riqueza lingüística de cada idioma."},
        {"role": "user", "content": f"{example['instruction']}\n\n{example['input']}"},
        {"role": "assistant", "content": example['output']},
    ]
    return {"messages": messages}

# Formatear datasets
train_formatted = [format_example(ex) for ex in train_data]
val_formatted = [format_example(ex) for ex in val_data]

# Verificar formato
print("Ejemplo formateado:")
for msg in train_formatted[0]["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:80]}...")


In [ ]:
# ============================================================
# 6. TOKENIZAR DATASET
# ============================================================
from unsloth.chat_templates import standardize_sharegpt, train_on_responses_only
from datasets import Dataset

def apply_chat_template(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

train_dataset = train_dataset.map(apply_chat_template, batched=True)
val_dataset = val_dataset.map(apply_chat_template, batched=True)

# Verificar longitudes de secuencia
lengths = [len(tokenizer.encode(t)) for t in train_dataset["text"][:500]]
print(f"Longitud promedio: {sum(lengths)/len(lengths):.0f} tokens")
print(f"Max: {max(lengths)}, Min: {min(lengths)}")
print(f"% > 2048 tokens: {100*sum(1 for l in lengths if l > 2048)/len(lengths):.1f}%")


In [ ]:
# ============================================================
# 7. ENTRENAR CON SFTTrainer
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir=CONFIG["output_dir"],
        per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        learning_rate=CONFIG["learning_rate"],
        warmup_ratio=CONFIG["warmup_ratio"],
        num_train_epochs=CONFIG["num_train_epochs"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        seed=CONFIG["seed"],
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
    ),
)

# Entrenar solo en respuestas del asistente (no en instrucciones)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

print("🚀 Iniciando entrenamiento...")
stats = trainer.train()
print(f"\n✅ Entrenamiento completado!")
print(f"   Tiempo total: {stats.metrics['train_runtime']/60:.1f} minutos")
print(f"   Loss final: {stats.metrics['train_loss']:.4f}")


In [ ]:
# ============================================================
# 8. EVALUACIÓN RÁPIDA — TRADUCCIONES DE PRUEBA
# ============================================================
FastLanguageModel.for_inference(model)

test_cases = [
    ("Traduce del español al K'iche'.", "Buenos días, ¿cómo estás?"),
    ("Traduce del español al Mam.", "La tierra es sagrada para nosotros."),
    ("Traduce del español al Q'eqchi'.", "Los niños están jugando en el campo."),
    ("Traduce del K'iche' al español.", "Saqarik, la utz awach?"),
    ("Traduce del español al Tz'utujil.", "El maíz ya está listo para cosechar."),
    ("Responde en K'iche' como un asistente amigable.", "¿Qué significa la palabra 'saqarik'?"),
]

for instruction, input_text in test_cases:
    messages = [
        {"role": "system", "content": "Eres MayaVoice, un asistente especializado en idiomas mayas de Guatemala."},
        {"role": "user", "content": f"{instruction}\n\n{input_text}"},
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=128, temperature=0.3,
        use_cache=True, do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"📝 {instruction}")
    print(f"   Input:  {input_text}")
    print(f"   Output: {response}")
    print()


In [ ]:
# ============================================================
# 9. MÉTRICAS BASELINE — BLEU y chrF (Issue #7)
# ============================================================
import sacrebleu
from collections import defaultdict

# Cargar test set con metadata
test_meta = load_jsonl(f"{DATA_DIR}/test_with_meta.jsonl")
print(f"Test set: {len(test_meta)} ejemplos")

# Generar traducciones para el test set (sample para velocidad)
SAMPLE_SIZE = 200  # Ajustar según tiempo disponible
random.seed(42)
test_sample = random.sample(test_meta, min(SAMPLE_SIZE, len(test_meta)))

FastLanguageModel.for_inference(model)

results_by_lang = defaultdict(lambda: {"refs": [], "hyps": [], "direction": []})

for i, example in enumerate(test_sample):
    messages = [
        {"role": "system", "content": "Eres MayaVoice, un asistente especializado en idiomas mayas de Guatemala."},
        {"role": "user", "content": f"{example['instruction']}\n\n{example['input']}"},
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=128, temperature=0.1,
        use_cache=True, do_sample=False,
    )
    
    hyp = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    ref = example["output"].strip()
    lang = example.get("_lang", "unknown")
    direction = example.get("_direction", "unknown")
    
    results_by_lang[lang]["refs"].append(ref)
    results_by_lang[lang]["hyps"].append(hyp)
    results_by_lang[lang]["direction"].append(direction)
    
    if (i + 1) % 50 == 0:
        print(f"  Procesados {i+1}/{len(test_sample)}...")

print(f"\n{'='*70}")
print(f"{'MÉTRICAS BASELINE — MayaVoice Sprint 1':^70}")
print(f"{'='*70}")
print(f"{'Idioma':<15} {'N':<6} {'BLEU':<10} {'chrF':<10}")
print("-" * 41)

all_refs, all_hyps = [], []
for lang in sorted(results_by_lang.keys()):
    data = results_by_lang[lang]
    refs = data["refs"]
    hyps = data["hyps"]
    all_refs.extend(refs)
    all_hyps.extend(hyps)
    
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    print(f"{lang:<15} {len(refs):<6} {bleu.score:<10.2f} {chrf.score:<10.2f}")

# Overall
bleu_all = sacrebleu.corpus_bleu(all_hyps, [all_refs])
chrf_all = sacrebleu.corpus_chrf(all_hyps, [all_refs])
print("-" * 41)
print(f"{'TOTAL':<15} {len(all_refs):<6} {bleu_all.score:<10.2f} {chrf_all.score:<10.2f}")

# Guardar resultados
metrics = {
    "model": CONFIG["base_model"],
    "overall_bleu": bleu_all.score,
    "overall_chrf": chrf_all.score,
    "sample_size": len(test_sample),
    "per_language": {}
}
for lang in sorted(results_by_lang.keys()):
    data = results_by_lang[lang]
    bleu = sacrebleu.corpus_bleu(data["hyps"], [data["refs"]])
    chrf = sacrebleu.corpus_chrf(data["hyps"], [data["refs"]])
    metrics["per_language"][lang] = {
        "bleu": bleu.score, "chrf": chrf.score, "n": len(data["refs"])
    }

with open("baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\n✅ Métricas guardadas en baseline_metrics.json")


In [ ]:
# ============================================================
# 10. GUARDAR MODELO
# ============================================================

# Opción A: Guardar LoRA adapters localmente
model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"✅ LoRA adapters guardados en {CONFIG['output_dir']}/")

# Opción B: Subir a HuggingFace Hub (descomentar para usar)
# model.push_to_hub(CONFIG["hub_model_id"], token="hf_YOUR_TOKEN")
# tokenizer.push_to_hub(CONFIG["hub_model_id"], token="hf_YOUR_TOKEN")
# print(f"✅ Modelo subido a https://huggingface.co/{CONFIG['hub_model_id']}")

# Opción C: Exportar a GGUF para Ollama (descomentar para usar)
# model.save_pretrained_gguf(
#     "mayavoice-gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
# )
# print("✅ GGUF exportado para uso con Ollama")


## 📊 Resumen del Sprint 1

### Qué se logró:
- ✅ Fine-tuning de Llama 3.1-8B con QLoRA (4-bit) usando Unsloth
- ✅ Dataset: 18,980 train / 2,360 val / 2,398 test (12 idiomas mayas)
- ✅ Formato: Instrucciones en español con chat template de Llama 3.1
- ✅ Métricas baseline BLEU y chrF por idioma

### Próximos pasos (Sprint 2):
- [ ] Conseguir datos de Kaqchikel (3er idioma más hablado, ~1M hablantes)
- [ ] Implementar RAG pipeline con ChromaDB
- [ ] Agregar datos de Achi e Ixil
- [ ] Crear test set manual con hablantes nativos (UVG)
- [ ] Probar Qwen 2.5-7B como alternativa si BLEU < esperado

### Issues cerrados con este notebook:
- [#6](https://github.com/DanielRegaladoUMiami/mayavoice-llm/issues/6): Primer fine-tuning con Unsloth
- [#7](https://github.com/DanielRegaladoUMiami/mayavoice-llm/issues/7): Métricas baseline BLEU y chrF
